# Module 2 — Working with the Data (in Pandas)

Homework: [cohorts/2026/homework2.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework2.md)
Submit: [courses.datatalks.club/sma-zoomcamp-2026/homework/hw02](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw02)

Kernel: **Python (stock-markets-zoomcamp)** — the shared repo venv.

| Q | Question | Answer |
| --- | --- | --- |
| 1 | Total withdrawn IPO value of the largest company class | **$500M** (Acquisition Corp, $499.99M) |
| 2 | Median Sharpe ratio on 2026-09-11 for pre-Sep-2025 IPOs | **0.04** (computed 0.0501) |
| 3 | Holding period maximising median growth | **1 month** (median 0.935) |
| 4 | Net income from the RSI < 30 strategy | **$65k** ($65,805 over 5,206 trades) |
| 5 | How to make an IPO strategy profitable | *free text — see Q5* |

> ⚠️ **The two iposcoop tables are live and change daily.** This notebook was run on
> **2026-09-18**, a week after the homework's 2026-09-11 anchor date, so raw row counts
> drift from the ones quoted in the task. Where that happens it is called out in the
> cell below the number. None of the drift changes a multiple-choice answer.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import io

import numpy as np
import pandas as pd
import requests

import smaz
from smaz import data, utils

utils.set_plot_defaults()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

AS_OF = pd.Timestamp("2026-09-11")   # the date the homework anchors on

# iposcoop blocks the default pandas/requests user-agent, exactly like Wikipedia in module 1.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


def read_html_table(url: str, index: int = 0) -> pd.DataFrame:
    """`pd.read_html` behind a browser user-agent."""
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    return pd.read_html(io.StringIO(response.text))[index]


print("smaz", smaz.__version__, "| repo root:", smaz.config.REPO_ROOT)

---

## Question 1 — [IPO] Withdrawn IPOs by company type

> **What is the total withdrawn IPO value (in $ millions) for the company class with the
> highest total withdrawal value?**
>
> From the [recently-filed IPO list](https://www.iposcoop.com/ipos-recently-filed/), find
> which company type saw the most withdrawn IPO value before Sep 11, 2026.

### Load the recently-filed table

In [ ]:
ipos_recent = read_html_table("https://www.iposcoop.com/ipos-recently-filed/")

print(ipos_recent.shape)
print(ipos_recent["Expected To Trade"].value_counts().to_string())
ipos_recent.head()

### Keep the withdrawn deals filed before the anchor date

The task expects **32** withdrawn rows. The live table now shows 34 — two more IPOs
(C2 Capital Group, OTSAW) were withdrawn in the week *after* the homework's 2026-09-11
cut-off. Filtering `File Date < 2026-09-11` reproduces the expected 32 exactly, so the
extra rows are genuinely new data rather than a parsing difference.

In [ ]:
withdrawn = ipos_recent[ipos_recent["Expected To Trade"] == "Withdrawn"].copy()
withdrawn["File Date"] = pd.to_datetime(withdrawn["File Date"])

print("withdrawn in the live table:", len(withdrawn))

withdrawn = withdrawn[withdrawn["File Date"] < AS_OF].copy()
print("withdrawn filed before", AS_OF.date(), ":", len(withdrawn))

### Classify the company type

The rules are applied **in order** and the first match wins, which is what makes
`EUPEC International Group Ltd.` a `Group` rather than a `Limited`. Matching is a plain
case-sensitive substring test, so `Xinxu Copper Industry **Technology** Ltd.` does *not*
hit the `Technologies` rule and falls through to `Limited`.

In [ ]:
COMPANY_TYPE_RULES = [
    ("Technologies", ["Technologies"]),
    ("Acquisition Corp", ["Acquisition Corp", "Acquisition Corporation", "Corp"]),
    ("Inc.", ["Inc", "Incorporated"]),
    ("Group", ["Group"]),
    ("Limited", ["Ltd", "Limited"]),
    ("Holdings", ["Holdings", "Holding"]),
]


def classify_company(name: str) -> str:
    """First matching rule wins -- the ordering above is part of the specification."""
    for label, patterns in COMPANY_TYPE_RULES:
        if any(p in name for p in patterns):
            return label
    return "Other"


withdrawn["Company Type"] = withdrawn["Company"].apply(classify_company)
print(withdrawn["Company Type"].value_counts().to_string())

# the two worked examples from the task description
for name in ["EUPEC International Group Ltd.", "Xinxu Copper Industry Technology Ltd."]:
    print(f"{name!r:45s} -> {classify_company(name)}")

### Parse prices and volumes

`'$8.00'` -> `8.0`, `'-'` and blanks -> `NaN`. `Avg_price` is the midpoint of the low/high
range (`.mean(axis=1)` skips NaNs, so a row with only one side still gets a price).

In [ ]:
def to_number(value) -> float:
    """'$1,234.50' -> 1234.5 ; '-' / '' / NaN -> NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value).replace("$", "").replace(",", "").strip()
    if text in {"-", ""}:
        return np.nan
    try:
        return float(text)
    except ValueError:
        return np.nan


withdrawn["Avg_price"] = withdrawn[["Price Low", "Price High"]].map(to_number).mean(axis=1)
withdrawn["Shares (millions)"] = withdrawn["Shares (millions)"].map(to_number)
withdrawn["Est $ Vol (millions)"] = withdrawn["Est $ Vol (millions)"].map(to_number)

withdrawn[["Company", "Shares (millions)", "Price Low", "Price High", "Avg_price"]].head()

### Deal value, with the estimated-volume fallback

Six rows carry `Shares (millions) == 0` *and* no price range at all, so the product is
`NaN` and the `Est $ Vol (millions)` column takes over. Note the fallback keys off the
*product* being null, not off the shares being zero — a genuine `0 x price = 0` would be
kept as zero.

In [ ]:
shares_x_price = withdrawn["Shares (millions)"] * withdrawn["Avg_price"]
withdrawn["Shares_offered_value"] = shares_x_price.where(
    shares_x_price.notna(), withdrawn["Est $ Vol (millions)"]
)

print("rows falling back to Est $ Vol:", int(shares_x_price.isna().sum()))
withdrawn[["Company", "Company Type", "Shares_offered_value"]].sort_values(
    "Shares_offered_value", ascending=False
).head(10)

In [ ]:
by_type = (
    withdrawn.groupby("Company Type")["Shares_offered_value"]
    .agg(total="sum", deals="count")
    .sort_values("total", ascending=False)
)
print(by_type.round(2).to_string())

winner = by_type.index[0]
print(f"\nhighest: {winner} -- ${by_type.loc[winner, 'total']:.2f}M")

In [ ]:
ax = by_type["total"].plot(kind="bar", color="#4C78A8", edgecolor="none")
ax.bar(winner, by_type.loc[winner, "total"], color="#E45756")  # highlight the winner
ax.set_title("Withdrawn IPO value by company type (filed before 2026-09-11)")
ax.set_xlabel("company type")
ax.set_ylabel("total value, $M")
for i, v in enumerate(by_type["total"]):
    ax.text(i, v + 6, f"{v:,.0f}", ha="center")

**Answer 1: Acquisition Corp, $499.99M — i.e. the `500` option.**

Five withdrawn SPAC-style shells add up to ~$500M, ahead of `Inc.` ($351M, a single deal —
Clear Street Group) and `Holdings` ($312M). Two observations worth keeping:

- The `Acquisition Corp` bucket wins on **count x uniform ticket size**, not on one big
  deal: SPACs all price at exactly $10.00, and four of the five offered 6–20M units.
  The `Inc.` bucket is a single company that happens to be large.
- The classification is doing real work here. `Helio Corp.` is a $15M uplisting unit deal
  with nothing SPAC-like about it, but the bare `Corp` pattern sweeps it into
  `Acquisition Corp` anyway. Drop it and the bucket is $485M — still the winner, so the
  answer is not sensitive to that edge case.

---

## Question 2 — [IPO] Median Sharpe ratio for 2025 IPOs

> **What is the median Sharpe ratio (as of 11 September 2026) for companies that went
> public before 1 September 2025?**

### Load the 2025 pricings list

In [ ]:
ipos_2025 = read_html_table("https://www.iposcoop.com/2025-pricings/")
ipos_2025["Offer Date"] = pd.to_datetime(ipos_2025["Offer Date"], format="%m/%d/%Y")
ipos_2025["Return_pct"] = ipos_2025["Return"].str.rstrip("%").astype(float)

print("IPOs priced in 2025:", len(ipos_2025))
ipos_2025.head()

### Filter to the pre-September cohort

A `0.00%` return on iposcoop means the *Current Price* column was never updated away from
the first-day close — the ticker is dead, halted or simply untracked, so those rows are
dropped as the task asks.

The task quotes **148** survivors; this run gets **146**. Two more names went stale in the
week since (iposcoop freezes the price rather than deleting the row), which is the same
live-data drift as in Q1. Two tickers out of 146 cannot move a median meaningfully.

In [ ]:
pre_sep = ipos_2025[ipos_2025["Offer Date"] < "2025-09-01"]
print("priced before 2025-09-01:", len(pre_sep))

ipo_universe = pre_sep[pre_sep["Return_pct"] != 0].copy()
print("after dropping 0% returns:", len(ipo_universe), "(task quotes 148)")

tickers = sorted(ipo_universe["Symbol"].unique())
print("unique tickers:", len(tickers))

### Download the daily OHLCV

Tickers are fetched one at a time rather than as a single batch: a third of this universe
is delisted micro-caps, and yfinance's batch mode silently returns an all-NaN block for a
dead ticker instead of telling you which one failed. The whole result is memoised to
`data/cache/` through `smaz.data.cached`, so re-running the notebook does not re-hit Yahoo.

In [ ]:
def download_ipo_ohlcv() -> pd.DataFrame:
    import warnings

    import yfinance as yf

    frames, failed = [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for ticker in tickers:
            try:
                hist = yf.download(
                    ticker,
                    start="2025-01-01",
                    end="2026-09-13",
                    auto_adjust=True,
                    progress=False,
                    threads=False,
                )
            except Exception:                      # noqa: BLE001 - yfinance raises broadly
                failed.append(ticker)
                continue
            if hist is None or hist.empty:
                failed.append(ticker)
                continue
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.droplevel(1)   # single ticker -> drop the level
            hist = hist.reset_index()
            hist["Ticker"] = ticker
            frames.append(hist)

    print(f"downloaded {len(frames)} tickers, {len(failed)} unavailable: {failed}")
    return pd.concat(frames, ignore_index=True)


stocks_df = data.cached(
    "ipo2025_ohlcv", download_ipo_ohlcv, tickers=tickers, end="2026-09-13"
)
stocks_df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(stocks_df.shape)
print("tickers with data:", stocks_df["Ticker"].nunique(), "(task quotes ~134)")
print("date range:", stocks_df["Date"].min().date(), "->", stocks_df["Date"].max().date())
stocks_df.head()

### Feature engineering

Both rolling calculations are `groupby("Ticker").transform(...)`, never a bare
`stocks_df["Close"].rolling(...)`. The frame is a stack of 132 separate price series; a
global rolling window would blend the tail of one ticker into the head of the next.

One caveat on the volatility formula the task prescribes: `Close.rolling(30).std()` is the
standard deviation of the **price**, in dollars, not of returns. So `volatility` scales
with the share price and the resulting `Sharpe` is not a textbook Sharpe ratio (a $200
stock looks far "riskier" than a $2 stock at identical percentage swings). It is
reproduced exactly as specified, but the level of the number is not comparable to a Sharpe
ratio computed from returns.

In [ ]:
close_by_ticker = stocks_df.groupby("Ticker")["Close"]

stocks_df["growth_252d"] = close_by_ticker.transform(lambda s: s / s.shift(252))
stocks_df["volatility"] = close_by_ticker.transform(
    lambda s: s.rolling(30).std() * np.sqrt(252)
)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

stocks_df[["Date", "Ticker", "Close", "growth_252d", "volatility", "Sharpe"]].tail()

### The 2026-09-11 snapshot

In [ ]:
snapshot = stocks_df[stocks_df["Date"] == AS_OF]
print("stocks trading on", AS_OF.date(), ":", len(snapshot))
print("reached the 252-day milestone:", int(snapshot["growth_252d"].notna().sum()))

snapshot[["growth_252d", "volatility", "Sharpe"]].describe()

In [ ]:
median_sharpe = snapshot["Sharpe"].median()
print(f"median Sharpe      : {median_sharpe:.4f}")
print(f"median growth_252d : {snapshot['growth_252d'].median():.4f}")
print(f"mean   growth_252d : {snapshot['growth_252d'].mean():.4f}")

# a handful of tickers have not moved for 30 sessions -> ~zero price-stdev -> exploding Sharpe
degenerate = snapshot[snapshot["volatility"] < 1e-4].dropna(subset=["Sharpe"])
print("\ntickers with an essentially flat 30-session price:", len(degenerate))
print(
    degenerate[["Ticker", "Close", "growth_252d", "volatility", "Sharpe"]].to_string(index=False)
)
print("\nmean Sharpe including them:", snapshot["Sharpe"].mean())
print("mean Sharpe excluding them :", snapshot.loc[snapshot["volatility"] >= 1e-4, "Sharpe"].mean())

**Answer 2: median Sharpe = 0.0501 → the `0.04` option** (the nearest of the four offered;
`0.1` is twice as far away).

What the describe table actually says:

- **`growth_252d` median 0.59 vs mean 1.06.** The typical 2025 IPO is worth **41% less**
  than a year ago, while the mean sits near break-even — one 33x survivor and a couple of
  10x names drag the average up over 130 stocks. Mean growth is useless here; this is the
  textbook case for the median.
- **130 of 132 stocks cleared the 252-day milestone**, so the sample is not being thinned
  by young listings — the losses are real, not a windowing artifact.
- **Three tickers have an infinite or absurd Sharpe** (EFTY, MAMK, MAGH) purely because
  their price has not moved for 30 consecutive sessions, making the denominator zero.
  They are a data-quality tell, not attractive risk-adjusted returns. The median is
  immune to them; a mean Sharpe is literally `inf`.
- Risk-adjusted ranking does beat raw growth in one respect: the best *finite* Sharpe
  names (HCMAU, CEPF — SPAC units grinding from $10.00 to $10.47) are boring, not the
  high-growth names. A 3.4% move with near-zero variance scores better than a 5x with
  violent swings.

---

## Question 3 — [IPO] Fixed-months holding strategy

> **What is the optimal number of months (1 to 12) to hold a newly IPO'd stock to maximise
> the median growth value?**

12 forward-looking columns, 21 trading days to the month, measured from each stock's own
first close.

In [ ]:
GROWTH_COLS = []
for month in range(1, 13):
    col = f"future_growth_{month}_m"
    stocks_df[col] = stocks_df.groupby("Ticker")["Close"].transform(
        lambda s, days=21 * month: s.shift(-days) / s
    )
    GROWTH_COLS.append(col)

stocks_df[["Date", "Ticker", "Close", *GROWTH_COLS[:3]]].head()

### Isolate each stock's first trading day

The inner join keeps exactly one row per ticker — the IPO day — so every growth column is
now "what a buyer at the first close would have earned".

In [ ]:
min_dates = stocks_df.groupby("Ticker")["Date"].min().reset_index()
ipo_day = min_dates.merge(stocks_df, on=["Ticker", "Date"], how="inner")

print("rows after the inner join:", len(ipo_day))
ipo_day[["Ticker", "Date", "Close", *GROWTH_COLS[:3]]].head()

In [ ]:
horizon_stats = ipo_day[GROWTH_COLS].describe().T[["count", "mean", "50%"]]
horizon_stats.columns = ["count", "mean", "median"]
print(horizon_stats.round(3).to_string())

best_month = ipo_day[GROWTH_COLS].median().idxmax()
best_median = ipo_day[GROWTH_COLS].median().max()
print(f"\nbest: {best_month} -- median growth {best_median:.4f}")

In [ ]:
medians = ipo_day[GROWTH_COLS].median()
medians.index = range(1, 13)

ax = medians.plot(marker="o", color="#4C78A8")
ax.axhline(1.0, color="#E45756", linestyle="--", linewidth=1)
ax.text(11.4, 1.005, "break-even", color="#E45756", ha="right")
ax.set_title("Median growth of a 2025 IPO vs holding period (bought at the first close)")
ax.set_xlabel("holding period, months")
ax.set_ylabel("median growth multiple")
ax.set_xticks(range(1, 13))

### Robustness: the mean column is unusable, and one "IPO day" is not an IPO day

The `mean` column above peaks at **95x**, which is nonsense. The cause is `PPCB`
(Propanc Biopharma): a sub-penny stock at $0.02 that did a large reverse split, so the
ratio comes out at 12,500x. Two separate problems collapse into that one row:

1. **Reverse splits are not adjusted in this history**, so the "growth" is a corporate
   action, not a return.
2. **`min_date` is not the IPO date for uplistings.** Five names in this universe
   (PPCB, AVBH, ALM, CIIT, CAPS) already traded OTC before moving to NASDAQ, so their
   first row is just the start of the download window in January 2025 — months before the
   offer date.

Re-anchoring entry on the first session **on or after the actual offer date** fixes both.

In [ ]:
offer_dates = ipo_universe[["Symbol", "Offer Date"]].rename(
    columns={"Symbol": "Ticker"}
)
with_offer = stocks_df.merge(offer_dates, on="Ticker")
post_offer = with_offer[with_offer["Date"] >= with_offer["Offer Date"]]
entry_day = post_offer.loc[post_offer.groupby("Ticker")["Date"].idxmin()]

comparison = pd.DataFrame(
    {
        "median (min_date)": ipo_day[GROWTH_COLS].median(),
        "median (offer date)": entry_day[GROWTH_COLS].median(),
        "mean (min_date)": ipo_day[GROWTH_COLS].mean(),
        "mean (offer date)": entry_day[GROWTH_COLS].mean(),
    }
)
print(comparison.round(3).to_string())
print("\nbest month, re-anchored:", entry_day[GROWTH_COLS].median().idxmax())

**Answer 3: 1 month, median growth 0.9354 — the `1` option.**

The honest reading of that number: **the "optimal" holding period is the shortest one
offered, and it still loses 6.5%.** Every horizon is below 1.0, and the median decays
almost monotonically from 0.94 at one month to 0.48 at eleven — hold a median 2025 IPO for
a year and you lose **half your money**. The only kink is a small bounce at months 6 and
12, well inside the noise of 130 names.

For an investor the conclusion is not "buy IPOs and sell after a month", it is
**"buying the median IPO at the first close is a losing trade at every horizon, and the
best you can do by tuning the exit is lose less"**. The decay shape is what you would
expect from post-IPO drift: first-day pricing is set by demand in a supported book, and
that support decays as lockups roll off and coverage thins.

Re-anchoring on the true offer date leaves the answer unchanged (month 1, median 0.924)
but repairs the mean column — it drops from 95x to a believable 1.06x, and the medians
shift by 1–5 points. That is the version I would trust for anything beyond this homework.

---

## Question 4 — [Strategy] Simple RSI-based trading strategy

> **What is the total profit (in $ thousands) from investing $1000 every time a stock was
> oversold (RSI < 30)?**

In [ ]:
import gdown

FILE_ID = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
PARQUET_PATH = smaz.config.RAW_DIR / "module2_indicators.parquet"

if not PARQUET_PATH.exists():
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", str(PARQUET_PATH), quiet=False)

indicators = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
indicators["Date"] = pd.to_datetime(indicators["Date"])

print(indicators.shape)
print("tickers:", indicators["Ticker"].nunique())
print("date range:", indicators["Date"].min().date(), "->", indicators["Date"].max().date())
indicators[["Date", "Ticker", "Close_x", "rsi", "growth_future_30d", "ticker_type"]].head()

In [ ]:
RSI_THRESHOLD = 30
TRADE_SIZE = 1000

selected_df = indicators[
    (indicators["rsi"] < RSI_THRESHOLD)
    & (indicators["Date"] >= "2000-01-01")
    & (indicators["Date"] <= "2025-06-01")
]

net_income = TRADE_SIZE * (selected_df["growth_future_30d"] - 1).sum()

print(f"trades triggered     : {len(selected_df):,}")
print(f"avg 30-day return    : {(selected_df['growth_future_30d'] - 1).mean():.4%}")
print(f"win rate             : {(selected_df['growth_future_30d'] > 1).mean():.2%}")
print(f"capital deployed     : ${TRADE_SIZE * len(selected_df):,}")
print(f"net income           : ${net_income:,.2f}  ->  ${net_income / 1000:.1f} thousand")

The three sanity checks quoted in the task all reproduce exactly — 5,206 trades, a 1.26%
average 30-day return and a 55.13% win rate — so the dataset and the filter match the
intended ones.

In [ ]:
# where the trades come from, and whether the edge survives outside the biggest bucket
per_market = selected_df.groupby("ticker_type").apply(
    lambda g: pd.Series(
        {
            "trades": len(g),
            "avg_return": (g["growth_future_30d"] - 1).mean(),
            "win_rate": (g["growth_future_30d"] > 1).mean(),
            "net_income": TRADE_SIZE * (g["growth_future_30d"] - 1).sum(),
        }
    ),
    include_groups=False,
)
print(per_market.round(4).to_string())

**Answer 4: $65,806 — the `65` option.**

Reading past the headline:

- **The edge is real but tiny.** 1.26% per 30-day trade at a 55% win rate is a genuine
  mean-reversion signal, but it is an average over $5.2M of deployed capital for $66k of
  profit — a **1.26% return on each ticket**, not 66x anything. Quoting it as a dollar
  total flatters it.
- **Capital is ignored.** The $1,000-per-signal rule assumes unlimited, free capital and
  overlapping positions. 5,206 signals over 25 years means many concurrent open trades;
  the strategy is untradeable as stated without a position-sizing rule.
- **No costs, no slippage.** At 1.26% gross per trade, a round-trip cost of even 20bps on
  an illiquid oversold name eats ~16% of the edge.
- Loosening the threshold from 25 to 30 more than triples the opportunity count
  (~1,568 → 5,206). More signals at a similar per-trade edge is the right direction, but
  it also means RSI < 30 is a *weak* filter — it fires on ~2.3% of all bar-days.

---

## Question 5 — [Exploratory] Predicting a positive-return IPO

> Most IPO strategies deliver negative average **and** median returns (and even the 75th
> percentile). **How would you change the strategy to increase profitability?**

My starting instinct was to optimise the **Sharpe ratio** directly — mean-variance
analysis (Markowitz) for sizing, and **CAPM** or **APT** to model the expected return.
Working through it, the honest answer is that portfolio theory is the *second* problem
here, not the first, and it is worth being precise about why.

### Why CAPM/APT cannot be applied to an IPO on day one

Both models price expected return off **factor loadings**, and a loading is estimated by
regressing an asset's return history on factor returns. A company that listed this morning
has **no return history**, so:

- **CAPM** needs `beta` — unestimable at entry. You would need ~1–2 years of post-IPO
  data, by which point the trade in Q3 is already over.
- **APT** is strictly better suited than CAPM *in principle* (it admits multiple risk
  factors and makes no assumption about the market portfolio being efficient), but it has
  the same problem *and* one more: APT does not tell you what the factors are. You choose
  them, which turns it into the characteristic-based screen below anyway.

The workable substitute is a **cross-sectional characteristic model** — the Fama-French /
APT-in-practice approach: instead of estimating each IPO's loadings from its own history,
regress future returns on *observable deal characteristics* across the cohort, and price a
new IPO from its characteristics. That is testable with the data already in this notebook.

### Test: do deal characteristics separate winners from losers?

In [ ]:
deal_facts = ipos_2025.assign(
    offer_price=lambda d: d["Offer Price"].str.replace(r"[$,]", "", regex=True).astype(float),
    first_close=lambda d: d["1st Day Close"].str.replace(r"[$,]", "", regex=True).astype(float),
).rename(columns={"Symbol": "Ticker"})
deal_facts["deal_size_musd"] = deal_facts["Shares (millions)"] * deal_facts["offer_price"]
deal_facts["first_day_pop"] = deal_facts["first_close"] / deal_facts["offer_price"] - 1

cohort = ipo_day.merge(
    deal_facts[["Ticker", "Industry", "deal_size_musd", "first_day_pop"]], on="Ticker"
)
# PPCB is the reverse-split artifact diagnosed in Q3 -- excluded so it cannot dominate a mean
cohort = cohort[cohort["future_growth_12_m"].notna() & (cohort["Ticker"] != "PPCB")]
print("cohort:", len(cohort))

cohort["size_bucket"] = pd.qcut(cohort["deal_size_musd"], 3, labels=["small", "mid", "large"])
print(
    cohort.groupby("size_bucket", observed=True)[
        ["future_growth_1_m", "future_growth_6_m", "future_growth_12_m"]
    ]
    .median()
    .round(3)
    .to_string()
)
print(
    "\nspearman(deal size, 12m growth) :",
    round(cohort[["deal_size_musd", "future_growth_12_m"]].corr(method="spearman").iloc[0, 1], 3),
)
print(
    "spearman(first-day pop, 12m)    :",
    round(cohort[["first_day_pop", "future_growth_12_m"]].corr(method="spearman").iloc[0, 1], 3),
)

In [ ]:
# how much of the damage is concentrated in micro-cap deals?
for threshold in [0, 12, 25, 50, 100, 250]:
    g = cohort[cohort["deal_size_musd"] > threshold]["future_growth_12_m"]
    print(
        f"deal > ${threshold:4d}M  n={len(g):3d}  "
        f"median={g.median():.3f}  mean={g.mean():.3f}  "
        f"win rate={(g > 1).mean():5.1%}  sd={g.std():.2f}"
    )

In [ ]:
industry_stats = cohort.groupby("Industry")["future_growth_12_m"].agg(["median", "count"])
print(industry_stats[industry_stats["count"] >= 6].sort_values("median", ascending=False).round(3).to_string())

### What the data actually supports

**Deal size is the single strongest filter available at entry**, and it is knowable before
you trade — no return history required:

| Screen | n | median 12m | win rate | sd |
| --- | --- | --- | --- | --- |
| all IPOs | 129 | 0.475 | 27.1% | 2.15 |
| deal > $25M | 51 | 0.870 | 43.1% | 0.92 |
| deal > $100M | 41 | 0.842 | 41.5% | **0.56** |

Screening out sub-$25M deals lifts the median 12-month outcome from **0.48 to 0.87**,
raises the win rate from 27% to 43%, and — the part that matters for your Sharpe idea —
cuts the cross-sectional dispersion by **four-fifths** (sd 2.15 → 0.56). Industry adds a
second, weaker cut (Financials 0.82 and Health Care 0.68 vs Consumer Services 0.12), and
first-day pop is worthless as a predictor (Spearman 0.07 — chasing the pop does not work).

**But be clear about what this does and does not achieve.** Even the best screen leaves the
median below 1.0 and the equal-weight mean return at roughly −2% to −4% at every horizon.
The size filter removes most of the *loss* and most of the *variance*; it does not flip the
sign of the edge. On 129 names in a single cohort year, that is a hypothesis, not a result.

### So, the changes I would actually make

1. **Fix the entry filter before touching portfolio construction.** Exclude sub-$25M
   micro-cap listings and shell-like deals outright. That is the largest single
   improvement available and it is a one-line screen.
2. **Replace the "buy at first close" entry.** Q3 shows the decay starts immediately, so
   the first close is the worst possible entry. Two variants worth testing: buy at the
   *offer* price (requires allocation, which is the whole game in IPO investing), or wait
   out the lock-up expiry (~180 days) and buy the post-expiry washout instead.
3. **Use APT-style characteristics, not APT-style betas.** Build a cross-sectional model
   on observables available at listing — deal size, industry, underwriter tier, price
   revision vs the initial range, revenue/profitability, insider retention. This is
   module 3 work, and it is the right home for the "is this factor priced?" question.
4. **Then apply mean-variance sizing** — but on a filtered universe, and with the caveat
   that Markowitz needs a covariance matrix, which for 40 new listings with <1y of history
   will be badly conditioned. Equal-weighting the screened basket beats a fitted MVA
   portfolio at this sample size; shrinkage (Ledoit-Wolf) is the minimum bar before
   trusting optimiser weights.
5. **Consider that the correct answer may be "don't".** Across every screen, horizon and
   entry rule tried here, the 2025 IPO cohort's median outcome stayed negative. The Q4 RSI
   strategy — a boring mean-reversion rule on established names — has a positive expectancy
   and 5,206 independent observations behind it. That is a far better foundation for a
   Sharpe-optimised portfolio than a 129-name cohort where the base rate is a loss.

---

## Summary of answers

Copy these into `answers.md` and then into the submission form.

| Q | Question | Computed | Submitted option |
| --- | --- | --- | --- |
| 1 | Total withdrawn value of the largest class | Acquisition Corp, $499.99M | **500** |
| 2 | Median Sharpe ratio on 2026-09-11 | 0.0501 | **0.04** |
| 3 | Holding period maximising median growth | 1 month (median 0.9354) | **1** |
| 4 | Net income from the RSI < 30 strategy | $65,805.59 | **65** |
| 5 | Improving IPO strategy profitability | screen by deal size; characteristics over CAPM/APT betas | *free text* |